In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader

from ml.common import CAP, SEED, load_fd, select_sensors
from ml.sequences import make_windows
from ml.models import GRUReg

torch.manual_seed(SEED)
np.random.seed(SEED)
print("torch", torch.__version__)

WIN = 30
train, test, rul = load_fd("FD001")
keep = select_sensors(train)

# Neural networks need inputs on a similar scale: standardise with TRAIN statistics only
mean, std = train[keep].mean(), train[keep].std()
train_s, test_s = train.copy(), test.copy()
train_s[keep] = (train[keep] - mean) / std
test_s[keep] = (test[keep] - mean) / std
print(len(keep), "sensors")

torch 2.15.0+cpu
14 sensors


In [2]:
X_train, u_train = make_windows(train_s, keep, WIN)
y_train = train_s["RUL"].to_numpy(dtype=np.float32)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("NaNs:", int(np.isnan(X_train).sum()))

X_train: (20631, 30, 14) | y_train: (20631,)
NaNs: 0


In [3]:
eng = train_s[train_s["unit"] == 1].copy()
w_a, _ = make_windows(eng, keep, WIN)

future_wiped = eng.copy()
future_wiped.loc[future_wiped["cycle"] > 50, keep] = 0.0   # destroy everything after cycle 50
w_b, _ = make_windows(future_wiped, keep, WIN)

print("windows for cycles 1-50 unchanged:", np.allclose(w_a[:50], w_b[:50]))
print("window for cycle 51 changed:", not np.allclose(w_a[50], w_b[50]))

windows for cycles 1-50 unchanged: True
window for cycle 51 changed: True


In [4]:
X_test_all, u_test = make_windows(test_s, keep, WIN)
last_idx = test_s.reset_index(drop=True).groupby("unit").tail(1).index.to_numpy()
X_test = X_test_all[last_idx]
y_test = np.minimum(rul["RUL"].values, CAP).astype(np.float32)

print("X_test:", X_test.shape, "| y_test:", y_test.shape)
print("first engine ids:", u_test[last_idx][:5])
print("all test engines have >= 30 cycles (no padding in final windows):",
      bool((test.groupby("unit")["cycle"].max() >= WIN).all()))

X_test: (100, 30, 14) | y_test: (100,)
first engine ids: [1 2 3 4 5]
all test engines have >= 30 cycles (no padding in final windows): True


In [5]:
ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
loader = DataLoader(ds, batch_size=256, shuffle=True)
xb, yb = next(iter(loader))
print("batch:", tuple(xb.shape), tuple(yb.shape))

model = GRUReg(n_features=len(keep), hidden=64)
print(sum(p.numel() for p in model.parameters()), "trainable parameters")

with torch.no_grad():
    out = model(xb)
print("output:", tuple(out.shape))

batch: (256, 30, 14) (256,)
15425 trainable parameters
output: (256,)


## Findings: sequence windows for the GRU (FD001)

- One training sample is a window of 30 consecutive readings of the 14 kept sensors, with shape (30, 14). Its label is the capped RUL at the last cycle of the window. The full training set is (20631, 30, 14), one window per training row.
- For the first 29 cycles of an engine, the window is padded by repeating the engine's first reading. This gives every row a window, so the GRU is trained and cross-validated on exactly the same 20,631 rows as XGBoost, which keeps the comparison fair.
- Leakage check: I set every sensor reading after cycle 50 of engine 1 to zero. The windows for cycles 1-50 stayed identical, and only windows covering later cycles changed. So a window cannot see future readings.
- Test windows: each test engine contributes its last 30 cycles, giving (100, 30, 14). All test engines have at least 30 observed cycles (the shortest has 31), so none of the final test windows is padded.
- Inputs are standardised with the mean and standard deviation of the training engines, because a neural network trains poorly on features with very different scales (tree models like XGBoost don't need this). The test data is scaled with the training statistics, so no test information leaks in. In the cross-validation on Day 8, I will refit the scaler inside each fold to be fully strict.
- The GRU model reads the 30 readings in order and uses its last hidden state to predict RUL through a linear layer. It has 15,425 trainable parameters. An untrained forward pass returns one prediction per sample, shape (256,) for a batch of 256, so the shapes are correct.